# RO-078 DWA and Nav2's DWB controller: every number in the Note

All runs use `images/dwbkit.py`: our own Python version of DWB's algorithm, written from the navigation2 code on `main`
(commit 3286119, read 2026-10-12) with the parameter values of the Nav2 configuration guide's DWB example and our
robot's limits. It is not Nav2's software running. Nothing is random, so there is no seed. The six runs take about a
minute (cached after the first time).

In [1]:
import sys, math, collections
sys.path.insert(0, 'images')
import numpy as np
import dwbkit as d
r2, r3, r4 = (lambda x: round(float(x), 2)), (lambda x: round(float(x), 3)), (lambda x: round(float(x), 4))
R = d.results()
P = d.PARAMS
def summary(name):
    r = R[name]; c = r['cycles']
    xs = np.array([o['x'] for o in c])
    gp = (np.hypot(xs[:, 0] - d.PERSON[0], xs[:, 1] - d.PERSON[1]) - d.PERSON_R - d.R_BODY).min()
    go = d.gap(xs[:, 0], xs[:, 1]).min()
    end = c[-1]['x']
    return dict(reason=r['reason'], t_end=c[-1]['t'], t_arrive=r['t_arrive'], end=end,
                door_err=float(np.hypot(*(end[:2] - d.GOAL[:2]))), gap_person=float(gp), gap_map=float(go))
for k in R:
    s = summary(k)
    print(f"{k:12s} {s['reason']:28s} last t {s['t_end']:5.2f} s  end {np.round(s['end'], 3)}  "
          f"from door pose {r3(s['door_err'])} m  closest to person {r3(s['gap_person'])} m  to the map {r3(s['gap_map'])} m")

replan       goal reached                 last t 13.30 s  end [3.39  3.593 1.326]  from door pose 0.23 m  closest to person 0.002 m  to the map 0.145 m
once         failed to make progress      last t 12.45 s  end [2.748 2.253 1.511]  from door pose 1.51 m  closest to person 0.069 m  to the map 0.263 m
free         goal reached                 last t  9.35 s  end [3.381 3.42  1.327]  from door pose 0.233 m  closest to person -0.412 m  to the map 0.17 m
free_replan  goal reached                 last t  9.60 s  end [3.386 3.411 1.328]  from door pose 0.231 m  closest to person -0.412 m  to the map 0.164 m
spin6        failed to make progress      last t 15.50 s  end [3.382 3.449 1.236]  from door pose 0.224 m  closest to person -0.423 m  to the map 0.191 m
dwa          failed to make progress      last t 10.00 s  end [2.009 1.005 0.482]  from door pose 2.959 m  closest to person 1.51 m  to the map 0.741 m


## 1. The global plan and the parameters

In [2]:
plan0 = d.global_plan(d.START[:2])
print('first plan: corner points', d.ROUTE_PTS.tolist(), 'length', r3(d.path_length(plan0)), 'm,', len(plan0), 'poses')
cells, L = d.astar((4, 2), (7, 7))
print('RO-034 A* route between cell centres:', r3(L), 'm')
for k in ('controller_frequency', 'sim_time', 'vx_samples', 'vtheta_samples', 'linear_granularity', 'angular_granularity',
          'max_vel_x', 'max_vel_theta', 'acc_lim_x', 'acc_lim_theta', 'forward_point_distance', 'xy_goal_tolerance',
          'yaw_goal_tolerance', 'costmap_size', 'resolution', 'update_frequency', 'inflation_radius', 'cost_scaling_factor'):
    print(k, P[k])
print('scales', P['scales'])

first plan: corner points [[2.0, 1.0], [2.75, 1.75], [2.75, 3.25], [3.25, 3.75], [3.6, 3.5]] length 3.698 m, 77 poses
RO-034 A* route between cell centres: 3.414 m
controller_frequency 20.0
sim_time 1.7
vx_samples 20
vtheta_samples 20
linear_granularity 0.05
angular_granularity 0.025
max_vel_x 0.6
max_vel_theta 1.0
acc_lim_x 0.5
acc_lim_theta 3.2
forward_point_distance 0.1
xy_goal_tolerance 0.25
yaw_goal_tolerance 0.25
costmap_size 3.0
resolution 0.05
update_frequency 5.0
inflation_radius 0.7
cost_scaling_factor 3.0
scales {'RotateToGoal': 32.0, 'Oscillation': 1.0, 'BaseObstacle': 0.02, 'GoalAlign': 24.0, 'PathAlign': 32.0, 'PathDist': 32.0, 'GoalDist': 24.0, 'PreferForward': 1.0, 'Twirling': 0.0}


## 2. The person's cycle (t = 2.4 s, main run: plan remade every second)

In [3]:
C = R['replan']['cycles']
k48 = [i for i, o in enumerate(C) if abs(o['t'] - d.T_PERSON) < 1e-9][0]
o = C[k48]; rec = o['rec']
x, v0, w0 = o['x'], o['v'], o['w']
gap_p = math.hypot(*(x[:2] - d.PERSON)) - d.PERSON_R - d.R_BODY
print('cycle', k48, 't', o['t'], 'pose', np.round(x[:2], 3), 'heading', r2(math.degrees(x[2])), 'deg; speed', r3(v0),
      'm/s; turn rate', r3(w0), 'rad/s; gap to the person', r3(gap_p), 'm')
dt_c = 1 / P['controller_frequency']
print('one cycle:', dt_c, 's; DWA window v', r3(v0 - P['acc_lim_x'] * dt_c), 'to', r3(v0 + P['acc_lim_x'] * dt_c),
      '; w', r3(w0 - P['acc_lim_theta'] * dt_c), 'to', r3(w0 + P['acc_lim_theta'] * dt_c))
print('sim_time window: v', r3(v0 - P['acc_lim_x'] * P['sim_time']), 'to', r3(v0 + P['acc_lim_x'] * P['sim_time']),
      'clipped to', rec['vwin'], '; w', r3(w0 - P['acc_lim_theta'] * P['sim_time']), 'to',
      r3(w0 + P['acc_lim_theta'] * P['sim_time']), 'clipped to', rec['wwin'])
print('speeds', len(rec['vs']), np.round(rec['vs'], 4).tolist())
print('turn rates', len(rec['ws']), np.round(rec['ws'], 4).tolist())
n_all = len(rec['vs']) * len(rec['ws'])
n_wheel = sum(1 for v in rec['vs'] for w in rec['ws'] if not (v == 0 and w == 0) and not d.wheel_ok(v, w))
print('pairs', n_all, '- (0, 0) =', n_all - 1, '- wheel limit', n_wheel, '=', len(rec['pairs']))
print('example dropped: v 0.6, w 0.0526: right wheel', r4(0.6 + 0.0526 * 0.1), 'm/s')
print('legal', rec['n_legal'], 'illegal', len(rec['pairs']) - rec['n_legal'],
      collections.Counter(r.split(':')[0] for r in rec['reasons'] if r))
assert len(rec['pairs']) == 375 and rec['n_legal'] == 242

cycle 48 t 2.4 pose [2.637 1.756] heading 66.19 deg; speed 0.568 m/s; turn rate 0.263 rad/s; gap to the person 0.578 m
one cycle: 0.05 s; DWA window v 0.543 to 0.593 ; w 0.103 to 0.423
sim_time window: v -0.282 to 1.418 clipped to (0.0, 0.6) ; w -5.177 to 5.703 clipped to (-1.0, 1.0)
speeds 20 [0.0, 0.0316, 0.0632, 0.0947, 0.1263, 0.1579, 0.1895, 0.2211, 0.2526, 0.2842, 0.3158, 0.3474, 0.3789, 0.4105, 0.4421, 0.4737, 0.5053, 0.5368, 0.5684, 0.6]
turn rates 21 [-1.0, -0.8947, -0.7895, -0.6842, -0.5789, -0.4737, -0.3684, -0.2632, -0.1579, -0.0526, 0.0, 0.0526, 0.1579, 0.2632, 0.3684, 0.4737, 0.5789, 0.6842, 0.7895, 0.8947, 1.0]
pairs 420 - (0, 0) = 419 - wheel limit 44 = 375
example dropped: v 0.6, w 0.0526: right wheel 0.6053 m/s
legal 242 illegal 133 Counter({'BaseObstacle': 133})


In [4]:
# stopping: Fox's admissible speed and the rollout length
for vv in (0.6, v0):
    print('from', r3(vv), 'm/s at 0.5 m/s^2: stops in', r3(vv / 0.5), 's over', r3(vv ** 2 / (2 * 0.5)), 'm')
print('admissible speed for the', r3(gap_p), 'm gap: sqrt(2 x gap x 0.5) =', r3(math.sqrt(2 * gap_p * 0.5)), 'm/s')
# keep going (0.568, 0.263): BaseObstacle looks up each pose's cell and rejects the first of cost 253 or 254
cm_k = o['cm']
tr_k = d.rollout(x, (v0, w0), (v0, w0))
n_k, h_k = d.time_steps(v0, w0)
ik, jk, _ = cm_k.world_to_map(tr_k[:, 0], tr_k[:, 1])
k_bad = int(np.argmax(cm_k.cost[jk, ik] >= d.INSCRIBED))
print('keep going (0.568, 0.263):', n_k, 'steps of', r3(h_k), 's; pose', k_bad, 'is on a cell of cost',
      int(cm_k.cost[jk[k_bad], ik[k_bad]]), 'at', r2(k_bad * h_k), 's ahead')

from 0.6 m/s at 0.5 m/s^2: stops in 1.2 s over 0.36 m
from 0.568 m/s at 0.5 m/s^2: stops in 1.137 s over 0.323 m
admissible speed for the 0.578 m gap: sqrt(2 x gap x 0.5) = 0.76 m/s
keep going (0.568, 0.263): 20 steps of 0.085 s; pose 12 is on a cell of cost 253 at 1.02 s ahead


## 3. One rollout: the chosen pair

In [5]:
cmd = rec['cmd']
print('chosen pair', np.round(cmd, 4), 'total', r4(rec['total']))
n, dt = d.time_steps(*cmd)
print('steps: max(', r3(cmd[0] * 1.7 / 0.05), ',', r3(cmd[1] * 1.7 / 0.025), ') ->', n, 'steps of', r4(dt), 's')
tr = rec['traj']
vv = np.maximum(cmd[0], v0 - 0.5 * dt * np.arange(1, n + 1))
print('speed after each step', np.round(vv, 3).tolist())
k_reach = int(np.argmax(np.isclose(vv, cmd[0])))
print('speed reaches the sample after', k_reach + 1, 'steps =', r3((k_reach + 1) * dt), 's')
print('first poses', np.round(tr[:3], 4).tolist())
print('end pose', np.round(tr[-1], 3).tolist(), 'heading', r2(math.degrees(tr[-1, 2])), 'deg; travelled',
      r3(np.sum(np.hypot(*np.diff(tr[:, :2], axis=0).T))), 'm;', len(tr), 'poses')

chosen pair [0.1579 0.2632] total 26.0
steps: max( 5.368 , 17.895 ) -> 18 steps of 0.0944 s
speed after each step [0.521, 0.474, 0.427, 0.38, 0.332, 0.285, 0.238, 0.191, 0.158, 0.158, 0.158, 0.158, 0.158, 0.158, 0.158, 0.158, 0.158, 0.158]
speed reaches the sample after 9 steps = 0.85 s
first poses [[2.6373, 1.7561, 1.1552], [2.6572, 1.8012, 1.1801], [2.6743, 1.8426, 1.2049]]
end pose [2.744, 2.157, 1.603] heading 91.82 deg; travelled 0.418 m; 19 poses


## 4. The costmap the critics read

In [6]:
for dd in (0.2, 0.25, 0.3, 0.4, 0.5, 0.7, 0.75):
    print('distance', dd, 'm -> cost', d.inflation_cost(dd))
cm = o['cm'] if o.get('cm') is not None else rec.get('cm')
cm = C[k48]['cm']
print('costmap origin', r3(cm.ox), r3(cm.oy), 'cells', cm.n, 'x', cm.n)
i, j, ok = cm.world_to_map(tr[-1, 0], tr[-1, 1])
print('end of the chosen rollout: cell', int(i), int(j), 'cost', int(cm.cost[j, i]), 'distance to nearest obstacle',
      r3(cm.dist[j, i]), 'm')
print('cost check: 252 exp(-3 (', r3(cm.dist[j, i]), '- 0.25)) =', r2(252 * math.exp(-3 * (cm.dist[j, i] - 0.25))))

distance 0.2 m -> cost 253
distance 0.25 m -> cost 253
distance 0.3 m -> cost 216
distance 0.4 m -> cost 160
distance 0.5 m -> cost 119
distance 0.7 m -> cost 65
distance 0.75 m -> cost 0
costmap origin 1.1 0.25 cells 60 x 60
end of the chosen rollout: cell 32 38 cost 160 distance to nearest obstacle 0.401 m
cost check: 252 exp(-3 ( 0.401 - 0.25)) = 160.18


## 5. The critics on the chosen pair and on others

In [7]:
ms = cm.res * 0.5
print('map-grid scale factor: resolution x 0.5 =', ms)
eff = {k: (P['scales'][k] * (ms if k in ('GoalAlign', 'PathAlign', 'PathDist', 'GoalDist') else 1)) for k in P['critics']}
print('effective weights', {k: r4(v) for k, v in eff.items()})
loc = o['local']
print('local plan', len(loc), 'poses from', np.round(loc[0, :2], 3).tolist(), 'to', np.round(loc[-1, :2], 3).tolist(),
      'length', r3(d.path_length(loc)))
def show(idx):
    raw = rec['raw'][idx]
    parts = {k: raw[k] * eff[k] for k in raw}
    print(np.round(rec['pairs'][idx], 4).tolist(), 'raw', {k: r2(v) for k, v in raw.items()}, 'weighted',
          {k: r3(v) for k, v in parts.items()}, 'total', r3(rec['totals'][idx]))
tot = np.array([t if t is not None else np.nan for t in rec['totals']])
order = np.argsort(np.nan_to_num(tot, nan=1e9))
for i in order[:3]:
    show(i)
legal_fast = [i for i in range(len(tot)) if not np.isnan(tot[i]) and rec['pairs'][i][0] >= 0.5]
i_fast = min(legal_fast, key=lambda i: tot[i])
print('best legal pair at 0.5 m/s or more:'); show(i_fast)
i_str = [i for i, pq in enumerate(rec['pairs']) if abs(pq[0] - 0.5684) < 1e-3 and abs(pq[1] - 0.2632) < 1e-3][0]
print('keep going (0.568, 0.263):', rec['reasons'][i_str])
print('rank of the chosen pair among', rec['n_legal'], 'legal: 1; totals of the next', [r3(tot[i]) for i in order[1:4]])

map-grid scale factor: resolution x 0.5 = 0.025
effective weights {'RotateToGoal': 32.0, 'Oscillation': 1.0, 'BaseObstacle': 0.02, 'GoalAlign': 0.6, 'PathAlign': 0.8, 'PathDist': 0.8, 'GoalDist': 0.6}
local plan 30 poses from [2.608, 1.788] to [2.75, 3.2] length 1.433
[0.1579, 0.2632] raw {'RotateToGoal': 0.0, 'Oscillation': 0.0, 'BaseObstacle': 160.0, 'GoalAlign': 17.0, 'PathAlign': 0.0, 'PathDist': 0.0, 'GoalDist': 21.0} weighted {'RotateToGoal': 0.0, 'Oscillation': 0.0, 'BaseObstacle': 3.2, 'GoalAlign': 10.2, 'PathAlign': 0.0, 'PathDist': 0.0, 'GoalDist': 12.6} total 26.0
[0.1895, 0.2632] raw {'RotateToGoal': 0.0, 'Oscillation': 0.0, 'BaseObstacle': 160.0, 'GoalAlign': 17.0, 'PathAlign': 0.0, 'PathDist': 0.0, 'GoalDist': 21.0} weighted {'RotateToGoal': 0.0, 'Oscillation': 0.0, 'BaseObstacle': 3.2, 'GoalAlign': 10.2, 'PathAlign': 0.0, 'PathDist': 0.0, 'GoalDist': 12.6} total 26.0
[0.2526, 0.3684] raw {'RotateToGoal': 0.0, 'Oscillation': 0.0, 'BaseObstacle': 212.0, 'GoalAlign': 16.0, 

## 6. The three runs in the room

In [8]:
once = R['once']['cycles']
vs_once = np.array([c['v'] for c in once]); ts_once = np.array([c['t'] for c in once])
k_stop = int(np.argmax((vs_once < 1e-9) & (ts_once > 2.4)))
xo = once[k_stop]['x']
print('plan made once: the robot first stands still at', r2(ts_once[k_stop]), 's at', np.round(xo[:2], 3).tolist(),
      'gap to the person', r3(math.hypot(*(xo[:2] - d.PERSON)) - 0.45), 'm; aborted at', r2(once[-1]['t']), 's:',
      R['once']['reason'])
xs_once = np.array([c['x'] for c in once])
print('  closest to the person', r3((np.hypot(xs_once[:, 0] - d.PERSON[0], xs_once[:, 1] - d.PERSON[1]) - 0.45).min()), 'm')
plans = R['replan']['plans']
p3 = [p for t, p in plans if abs(t - 3.0) < 1e-9][0]
dd = np.diff(p3[:, :2], axis=0); turn = np.where(np.abs(np.diff(np.arctan2(dd[:, 1], dd[:, 0]))) > 1e-6)[0] + 1
print('new plan at 3.0 s: corners', np.round(p3[[0, *turn, -1], :2], 3).tolist(), 'length', r3(d.path_length(p3)))
C = R['replan']['cycles']
xs = np.array([c['x'] for c in C]); ts = np.array([c['t'] for c in C])
gpp = np.hypot(xs[:, 0] - d.PERSON[0], xs[:, 1] - d.PERSON[1]) - 0.45
print('main run: closest to the person', r3(gpp.min()), 'm at', r2(ts[np.argmin(gpp)]), 's')
nolegal = [c['t'] for c in C if c['rec'] is not None and c['cmd'] is not None and c['rec']['cmd'] is None]
print('cycles with no legal rollout (stop sent, within the 0.3 s patience):', [r2(t) for t in nolegal])
rot = [c['t'] for c in C if c.get('crit_state', (0, 0))[1]]
print('turning on the spot from', r2(min(rot)), 's to', r2(R['replan']['t_arrive']), 's; heading',
      r2(math.degrees(C[[i for i, c in enumerate(C) if c['t'] == min(rot)][0]]['x'][2])), 'deg ->',
      r2(math.degrees(C[-1]['x'][2])), 'deg')
s = summary('replan')
print('arrival', s['t_arrive'], 's,', r3(s['door_err']), 'm from the door pose, heading', r2(math.degrees(s['end'][2])), 'deg')
for name in ('free', 'free_replan'):
    s = summary(name); Cf = R[name]['cycles']
    rotf = [c['t'] for c in Cf if c.get('crit_state', (0, 0))[1]]
    print(name, 'arrival', s['t_arrive'], 's,', r3(s['door_err']), 'm from the door pose; turning on the spot from', r2(min(rotf)), 's')
print('top speed main run', r3(max(c['v'] for c in C)), 'm/s; speed at 3.0 s', r3([c['v'] for c in C if abs(c['t'] - 3.0) < 1e-9][0]),
      'm/s; top speed from 5.0 s to 9.7 s', r3(max(c['v'] for c in C if 5.0 <= c['t'] <= 9.7)), 'm/s')
xo = np.array([c['x'][:2] for c in R['once']['cycles']]); xr = np.array([c['x'][:2] for c in C])
n = min(len(xo), len(xr)); sep = np.hypot(*(xo[:n] - xr[:n]).T); tt = np.array([c['t'] for c in C])[:n]
print('largest gap between the two runs before 2.4 s', r3(sep[tt < 2.4].max()), 'm; at 3.0 s', r3(sep[np.isclose(tt, 3.0)][0]), 'm')
print('highest legal BaseObstacle cost 252 x 0.02 =', r2(252 * 0.02), '= PathDist cells', r2(252 * 0.02 / 0.8))
print('goal checker heading tolerance', r2(math.degrees(0.25)), 'deg; arrival heading error', r2(90 - math.degrees(C[-1]['x'][2])), 'deg')

plan made once: the robot first stands still at 6.75 s at [2.748, 2.252] gap to the person 0.071 m; aborted at 12.45 s: failed to make progress
  closest to the person 0.069 m
new plan at 3.0 s: corners [[2.722, 1.992], [2.25, 2.25], [2.25, 3.25], [2.75, 3.75], [3.25, 3.75], [3.6, 3.5]] length 3.176
main run: closest to the person 0.002 m at 4.75 s
cycles with no legal rollout (stop sent, within the 0.3 s patience): [4.5, 4.7, 4.75, 4.85, 4.95, 5.0]
turning on the spot from 9.7 s to 13.3 s; heading -13.85 deg -> 75.99 deg
arrival 13.3 s, 0.23 m from the door pose, heading 75.99 deg
free arrival 9.35 s, 0.233 m from the door pose; turning on the spot from 6.7 s
free_replan arrival 9.6 s, 0.231 m from the door pose; turning on the spot from 6.8 s
top speed main run 0.568 m/s; speed at 3.0 s 0.268 m/s; top speed from 5.0 s to 9.7 s 0.537 m/s


largest gap between the two runs before 2.4 s 0.0 m; at 3.0 s 0.003 m
highest legal BaseObstacle cost 252 x 0.02 = 5.04 = PathDist cells 6.3
goal checker heading tolerance 14.32 deg; arrival heading error 14.01 deg


## 7. Two settings that fail on our robot

In [9]:
# 20 turn-rate samples over our robot's full +-6 rad/s
w_spacing = (2 * 5.44) / 19
print('spin limit 6 rad/s: window +-', 3.2 * 1.7, 'rad/s around the turn rate, 20 samples ->', r3(w_spacing), 'rad/s apart')
s = summary('spin6'); print('spin6:', s['reason'], 'at', s['t_end'], 's; heading', r2(math.degrees(s['end'][2])),
      'deg, needs', r2(90 - math.degrees(P['yaw_goal_tolerance'])), 'deg or more')
hs = [math.degrees(c['x'][2]) for c in R['spin6']['cycles'] if c['t'] >= 10.0]
print('spin6 heading over its last 5.5 s: from', r2(min(hs)), 'to', r2(max(hs)), 'deg')
# DWA's one-cycle window from rest
c0 = R['dwa']['cycles'][0]['rec']
print('DWA from rest: v', np.round(c0['vwin'], 4).tolist(), 'w', np.round(c0['wwin'], 4).tolist(), 'pairs', len(c0['pairs']))
tot0 = np.array([t if t is not None else np.nan for t in c0['totals']])
print('lowest total', r3(np.nanmin(tot0)), 'shared by', int(np.sum(np.isclose(tot0, np.nanmin(tot0)))), 'pairs; chosen',
      np.round(c0['cmd'], 4).tolist())
print('farthest a rollout from rest reaches in 1.7 s:', r3(0.025 * 1.7), 'm, one cell is', P['resolution'], 'm')
s = summary('dwa'); print('dwa:', s['reason'], 'at', s['t_end'], 's; moved', r3(math.hypot(*(s['end'][:2] - d.START[:2]))), 'm')

spin limit 6 rad/s: window +- 5.44 rad/s around the turn rate, 20 samples -> 0.573 rad/s apart
spin6: failed to make progress at 15.5 s; heading 70.82 deg, needs 75.68 deg or more
spin6 heading over its last 5.5 s: from 60.25 to 73.09 deg
DWA from rest: v [0.0, 0.025] w [-0.16, 0.16] pairs 419
lowest total 50.4 shared by 136 pairs; chosen [0.0, 0.0084]
farthest a rollout from rest reaches in 1.7 s: 0.043 m, one cell is 0.05 m
dwa: failed to make progress at 10.0 s; moved 0.01 m
